# Señario CAS — Visualizador de Esqueleto 3D

Renderizado 3D de keypoints MediaPipe extraídos de videos del señario CAS.  
Pose + manos con profundidad (x, y, z). Rotación con el mouse.

**Uso:** correr todas las celdas → usar el slider para explorar la seña.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
DATA_DIR = Path('data')
WORD     = 'FEBRERO'   # ← cambiá acá para ver otra seña

with open(DATA_DIR / f'{WORD}_3d.json') as f:
    data = json.load(f)

frames = data['frames']
fps    = data['fps']
print(f"Seña: {WORD} | Frames: {len(frames)} | FPS: {fps}")

In [ ]:
# ── Conexiones del esqueleto ───────────────────────────────────────────────────
HAND_CONNECTIONS = [
    # Palma
    (0, 1), (0, 5), (0, 17), (5, 9), (9, 13), (13, 17),
    # Pulgar
    (1, 2), (2, 3), (3, 4),
    # Índice
    (5, 6), (6, 7), (7, 8),
    # Medio
    (9, 10), (10, 11), (11, 12),
    # Anular
    (13, 14), (14, 15), (15, 16),
    # Meñique
    (17, 18), (18, 19), (19, 20),
]

POSE_CONNECTIONS = [
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
]
POSE_LANDMARKS = [11, 12, 13, 14, 15, 16]

def get_hand_xyz(frame_data, hand='right'):
    lms = frame_data['landmarks_3d'][f'{hand}_hand']
    return np.array(lms)  # (21, 3)

def get_pose_xyz(frame_data):
    lms = frame_data['landmarks_3d']['pose']
    return np.array(lms)[:, :3]  # (33, 3) — descarta visibility

In [ ]:
# ── Función de dibujo 3D ───────────────────────────────────────────────────────
def draw_frame(ax, frame_data):
    ax.clear()

    # Fondo oscuro
    ax.set_facecolor('#1a1a2e')
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('#2a2a4e')
    ax.yaxis.pane.set_edgecolor('#2a2a4e')
    ax.zaxis.pane.set_edgecolor('#2a2a4e')
    ax.grid(False)
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)  # y invertido (0 arriba)
    ax.tick_params(colors='#444466')

    # Pose
    pose = get_pose_xyz(frame_data)
    if frame_data['pose_detected']:
        for a, b in POSE_CONNECTIONS:
            pa, pb = pose[a], pose[b]
            ax.plot([pa[0], pb[0]], [pa[1], pb[1]], [pa[2], pb[2]],
                    '-', color='#4a4a8a', lw=3)
        for i in POSE_LANDMARKS:
            ax.scatter(*pose[i], color='#6a6aaa', s=30, zorder=5)

    # Mano izquierda
    if frame_data['left_hand']:
        lh = get_hand_xyz(frame_data, 'left')
        for a, b in HAND_CONNECTIONS:
            ax.plot([lh[a][0], lh[b][0]], [lh[a][1], lh[b][1]], [lh[a][2], lh[b][2]],
                    '-', color='#00b4d8', lw=2)
        ax.scatter(lh[:, 0], lh[:, 1], lh[:, 2], c='#90e0ef', s=20, zorder=5)
        ax.scatter(*lh[0], c='white', s=40, zorder=6)

    # Mano derecha
    if frame_data['right_hand']:
        rh = get_hand_xyz(frame_data, 'right')
        for a, b in HAND_CONNECTIONS:
            ax.plot([rh[a][0], rh[b][0]], [rh[a][1], rh[b][1]], [rh[a][2], rh[b][2]],
                    '-', color='#ff6b6b', lw=2)
        ax.scatter(rh[:, 0], rh[:, 1], rh[:, 2], c='#ffa8a8', s=20, zorder=5)
        ax.scatter(*rh[0], c='white', s=40, zorder=6)

    n = frame_data['frame']
    t = n / fps
    ax.set_title(f'{WORD}  —  frame {n}  ({t:.2f}s)  |  conf: {frame_data["confidence"]:.2f}',
                 color='white', fontsize=11, pad=8)

    legend = [
        Line2D([0],[0], color='#ff6b6b', lw=2, label='Mano derecha'),
        Line2D([0],[0], color='#00b4d8', lw=2, label='Mano izquierda'),
        Line2D([0],[0], color='#4a4a8a', lw=2, label='Pose'),
    ]
    ax.legend(handles=legend, loc='lower right', fontsize=8,
              facecolor='#1a1a2e', labelcolor='white', framealpha=0.7)

In [ ]:
# ── Slider interactivo ─────────────────────────────────────────────────────────
%matplotlib widget

fig = plt.figure(figsize=(7, 7))
fig.patch.set_facecolor('#1a1a2e')
ax = fig.add_subplot(111, projection='3d')
plt.tight_layout()

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frames) - 1,
    step=1,
    description='Frame:',
    continuous_update=True,
    layout=widgets.Layout(width='600px')
)

play = widgets.Play(
    value=0,
    min=0,
    max=len(frames) - 1,
    step=1,
    interval=int(1000 / fps),
    description='Play',
)

widgets.jslink((play, 'value'), (slider, 'value'))

def on_frame_change(change):
    draw_frame(ax, frames[change['new']])
    fig.canvas.draw_idle()

slider.observe(on_frame_change, names='value')
draw_frame(ax, frames[0])

display(widgets.VBox([widgets.HBox([play, slider])]))
plt.show()